In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

from features import feature_engeneering, get_fe_parametrs
from folds import FOLDS, X, y

oof_lof = np.zeros(len(X))
synth_lof = []

for fold, (tr_idx, val_idx) in enumerate(FOLDS):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)

    X_tr_f = feature_engeneering(
        df=X_tr, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q,
        fti_quantile=fti_q, tt4_quantile=tt4_q
    )

    X_val_f = feature_engeneering(
        df=X_val, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q,
        tt4_quantile=tt4_q, fti_quantile=fti_q
    )

    X_tr_normal = X_tr_f[y_tr == 0]

    scaler = StandardScaler()
    scaler.fit(X_tr_normal)

    X_tr_scaled = scaler.transform(X_tr_normal)
    X_val_scaled = scaler.transform(X_val_f)

    X_synth = X_tr_normal.copy()
    for col in X_tr_normal.columns:
        X_synth[col] = np.random.permutation(X_synth[col].values)
    X_synth_scaled = scaler.transform(X_synth)

    lof = LocalOutlierFactor(
        n_neighbors=20,
        novelty=True,
        contamination=0.07,
        n_jobs=-1
    )

    lof.fit(X_tr_scaled)

    oof_lof[val_idx] = -lof.score_samples(X_val_scaled)
    synth_lof.extend(-lof.decision_function(X_synth_scaled))

pr_auc_lof = average_precision_score(y, oof_lof)
print(f'PR-AUC - {pr_auc_lof}')

np.save('oof_lof.npy', oof_lof)
np.save('synth_lof.npy', synth_lof)

PR-AUC - 0.5067294262725374


: 